# 02. Treino e inferencia

Notebook canonico para executar `SpeechT5 LoRA` e materializar amostras por checkpoint usando o mesmo contrato operacional do `MANUAL_EXECUCAO.md`.

In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    markers = ('pyproject.toml', 'README.md', 'scripts')
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    workspace = Path('/workspace')
    if workspace.exists():
        candidates.append(workspace.resolve())
        for child in sorted(workspace.iterdir()):
            if child.is_dir():
                candidates.append(child.resolve())
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do notebook.')


PROJECT_ROOT = find_project_root()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'


def resolve_path(path: str | Path) -> Path:
    candidate = Path(path)
    return candidate if candidate.is_absolute() else PROJECT_ROOT / candidate


def assert_path(path: str | Path, *, kind: str | None = None) -> Path:
    resolved = resolve_path(path)
    if not resolved.exists():
        raise FileNotFoundError(f'Path not found: {resolved}')
    if kind == 'file' and not resolved.is_file():
        raise FileNotFoundError(f'Expected file, found: {resolved}')
    if kind == 'dir' and not resolved.is_dir():
        raise FileNotFoundError(f'Expected directory, found: {resolved}')
    print(f'OK: {resolved}')
    return resolved


def run_cmd(args: list[object], env: dict[str, object] | None = None, check: bool = True) -> subprocess.CompletedProcess:
    cmd = [str(arg) for arg in args]
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items() if value is not None})
    print('+', shlex.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, env=merged_env, check=check, text=True)


def run_project_python(args: list[object], env: dict[str, object] | None = None) -> subprocess.CompletedProcess:
    assert_path(VENV_PYTHON, kind='file')
    return run_cmd([VENV_PYTHON, *args], env=env)


def preview_csv(path: str | Path, rows: int = 5, columns: list[str] | None = None) -> pd.DataFrame:
    resolved = assert_path(path, kind='file')
    frame = pd.read_csv(resolved)
    if columns:
        frame = frame.loc[:, columns]
    print(f'rows={len(frame)} columns={list(frame.columns)}')
    display(frame.head(rows))
    return frame


def show_tree(path: str | Path, max_depth: int = 3, max_entries: int = 60) -> None:
    root = assert_path(path)
    print(root)
    base_depth = len(root.parts)
    shown = 0
    for child in sorted(root.rglob('*')):
        depth = len(child.parts) - base_depth
        if depth > max_depth:
            continue
        rel = child.relative_to(root)
        suffix = '/' if child.is_dir() else ''
        print(f"{'  ' * depth}{rel}{suffix}")
        shown += 1
        if shown >= max_entries:
            print(f'... truncated after {max_entries} entries')
            break


print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'VENV_PYTHON={VENV_PYTHON}')


## Parametros editaveis

Ajuste apenas esta celula quando quiser trocar arquivos, restringir condicionais ou sobrescrever a taxa global de GPU.

In [ ]:
CONFIG_PATH = Path('configs/speecht5_minimal.yaml')
MANIFEST_PATH = Path('data/manifests/data_manifest.csv')
SAMPLES_PATH = Path('artifacts/evaluation/samples.csv')
CHECKPOINT_DIR = Path('artifacts/checkpoints/lora')
AUDIO_BASE_DIR = Path('artifacts/audio')
CONDITIONS: list[str] = []
GPU_HOURLY_RATE_OVERRIDE: float | None = None


## Validacoes de pre-requisito

In [ ]:
assert_path(VENV_PYTHON, kind='file')
assert_path(CONFIG_PATH, kind='file')
assert_path(MANIFEST_PATH, kind='file')
assert_path(SAMPLES_PATH, kind='file')
preview_csv(MANIFEST_PATH, columns=['speaker_id', 'utterance_id', 'split', 'duration_s'])
preview_csv(SAMPLES_PATH, columns=['sample_id', 'condition', 'speaker_id', 'prompt_id', 'status'])


## Executar treino LoRA e inferencia por checkpoint

In [ ]:
command: list[object] = [
    'scripts/run_speecht5_lora.py',
    '--config', CONFIG_PATH,
    '--manifest', MANIFEST_PATH,
    '--samples', SAMPLES_PATH,
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--audio-base-dir', AUDIO_BASE_DIR,
]

for condition in CONDITIONS:
    command.extend(['--condition', condition])

if GPU_HOURLY_RATE_OVERRIDE is not None:
    command.extend(['--gpu-hourly-rate', GPU_HOURLY_RATE_OVERRIDE])

run_project_python(command)


## Inspecionar checkpoints materializados

In [ ]:
assert_path(CHECKPOINT_DIR, kind='dir')
show_tree(CHECKPOINT_DIR, max_depth=4, max_entries=80)


## Resumo do samples.csv apos treino

In [ ]:
samples = preview_csv(
    SAMPLES_PATH,
    columns=['sample_id', 'condition', 'checkpoint_label', 'speaker_id', 'prompt_id', 'status', 'audio_path'],
)

materialized = samples[samples['checkpoint_label'].fillna('').astype(str).str.strip().ne('')].copy()
print(f'materialized_rows={len(materialized)}')
print(samples['status'].fillna('').value_counts(dropna=False))


## Listagem analitica por checkpoint, speaker e status

In [ ]:
summary = pd.read_csv(resolve_path(SAMPLES_PATH))
summary['checkpoint_label'] = summary['checkpoint_label'].fillna('').astype(str)
summary = summary[summary['checkpoint_label'].str.strip().ne('')].copy()
grouped = (
    summary.groupby(['checkpoint_label', 'speaker_id', 'status'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['checkpoint_label', 'speaker_id', 'status'])
)
display(grouped)


## Utilitarios avancados

Os notebooks canonicos nao executam cleanup destrutivo. Se precisar reprocessar uma condicao especifica, use manualmente os scripts utilitarios descritos no `MANUAL_EXECUCAO.md`, como `scripts/clear_speecht5_training_results.py` e `scripts/remove_speecht5_condition.py`.